# BS Environment Prompt Demo

This notebook shows the exact prompts the environment constructs across a short manual BS rollout.

It focuses on:
- the initial `PLAY` prompt
- the follow-up `CHALLENGE` prompt
- the next player's `PLAY` prompt after a pass


In [ ]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import importlib
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents] if (candidate / 'Environments').exists() and (candidate / 'LocalizationScripts').exists()), NOTEBOOK_ROOT)
ENV_SRC = REPO_ROOT / 'Environments' / 'BS' / 'src'
if str(ENV_SRC) not in sys.path:
    sys.path.insert(0, str(ENV_SRC))

import bs_environment as bs_env
importlib.reload(bs_env)

BSEnvironment = bs_env.BSEnvironment
print('Imported from:', ENV_SRC / 'bs_environment.py')


In [ ]:
class DemoAgent:
    def __init__(self, name):
        self.name = name
        self.reasoning_instruction = 'COD'
        self.play_format = 'default'
        self.challenge_format = 'default'
        self.hand = []

    def add_cards(self, cards):
        self.hand.extend(cards)

    def remove_cards(self, cards):
        for card in cards:
            if card in self.hand:
                self.hand.remove(card)


def make_agents():
    return [
        DemoAgent('Alice'),
        DemoAgent('Bob'),
        DemoAgent('Carol'),
        DemoAgent('Dave'),
    ]


def make_env(seed=0):
    return BSEnvironment(make_agents(), seed=seed)


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_state(env, include_system_prompt=True):
    state = env.get_state(include_system_prompt=include_system_prompt)
    print('phase:', state['phase'])
    print('active_player:', state['active_player'])
    print('current_rank:', env.current_rank)
    print('pile_size:', len(env.pile))
    print('player_hands:', {agent.name: list(agent.hand) for agent in env.agents})
    print('recent_history:', env.history[-5:])
    print()
    show_messages(state['messages'])
    return state


In [ ]:
env = make_env(seed=0)
state = show_state(env, include_system_prompt=True)


In [ ]:
truthful_play = env.get_truthful_action()
play_result = env.manual_step(truthful_play)

print('truthful play action:')
pprint(truthful_play)
print('\nplay result:')
pprint(play_result)
print('\nnext prompt (challenge):')
_ = show_state(env, include_system_prompt=True)


In [ ]:
challenge_result = env.manual_step({'Action': 'Pass'})
print('challenge action:')
pprint({'Action': 'Pass'})
print('\nchallenge result:')
pprint(challenge_result)
print('\nnext prompt (next player play phase):')
_ = show_state(env, include_system_prompt=True)
